In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!cp /content/drive/MyDrive/drone-object-detection/data_processed.zip /content/
!cp /content/drive/MyDrive/drone-object-detection/visdrone.yaml /content/

In [ ]:
!unzip data_processed.zip

In [ ]:
!nvidia-smi
!pip install -q ultralytics>=8.3.0

import ultralytics
print(f"Ultralytics version: {ultralytics.__version__}")

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo26n.pt')

results = model.train(
    data='visdrone.yaml',
    epochs=35,
    batch=32,
    imgsz=640,
    patience=10,
    save=True,
    project='runs/detect',
    name='yolo26n_visdrone',
    exist_ok=True,
    pretrained=True,
    optimizer='Adam',
    seed=42,
    mosaic=1.0,
    mixup=0.1,
    degrees=10.0,
    scale=0.5,
    device=0  # GPU
)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

results_csv = Path('/content/runs/detect/runs/detect/yolo26n_visdrone/results.csv')

if results_csv.exists():
    df = pd.read_csv(results_csv)

    df.columns = df.columns.str.strip()

    print("\nColonnes disponibles:")
    print(df.columns.tolist())

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    ax = axes[0, 0]
    if 'train/box_loss' in df.columns:
        ax.plot(df['epoch'], df['train/box_loss'], label='Box Loss', linewidth=2)
    if 'train/cls_loss' in df.columns:
        ax.plot(df['epoch'], df['train/cls_loss'], label='Class Loss', linewidth=2)
    if 'train/dfl_loss' in df.columns:
        ax.plot(df['epoch'], df['train/dfl_loss'], label='DFL Loss', linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Training Loss Evolution')
    ax.legend()
    ax.grid(alpha=0.3)

    ax = axes[0, 1]
    if 'metrics/mAP50(B)' in df.columns:
        ax.plot(df['epoch'], df['metrics/mAP50(B)'], label='mAP50', linewidth=2, color='green')
    if 'metrics/mAP50-95(B)' in df.columns:
        ax.plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP50-95', linewidth=2, color='blue')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('mAP')
    ax.set_title('Validation mAP Evolution')
    ax.legend()
    ax.grid(alpha=0.3)

    ax = axes[1, 0]
    if 'metrics/precision(B)' in df.columns:
        ax.plot(df['epoch'], df['metrics/precision(B)'], label='Precision', linewidth=2, color='orange')
    if 'metrics/recall(B)' in df.columns:
        ax.plot(df['epoch'], df['metrics/recall(B)'], label='Recall', linewidth=2, color='purple')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Score')
    ax.set_title('Precision & Recall Evolution')
    ax.legend()
    ax.grid(alpha=0.3)

    ax = axes[1, 1]
    if 'lr/pg0' in df.columns:
        ax.plot(df['epoch'], df['lr/pg0'], linewidth=2, color='red')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Learning Rate')
    ax.set_title('Learning Rate Schedule')
    ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()

    print("Saved: training_curves.png")


    print("\n" + "="*60)
    print("METRICS")
    print("="*60)

    last_epoch = df.iloc[-1]

    metrics = {
        'mAP50': last_epoch.get('metrics/mAP50(B)', 0),
        'mAP50-95': last_epoch.get('metrics/mAP50-95(B)', 0),
        'Precision': last_epoch.get('metrics/precision(B)', 0),
        'Recall': last_epoch.get('metrics/recall(B)', 0)
    }

    for name, value in metrics.items():
        print(f"{name:15s}: {value:.3f}")

else:
    print("no results.csv")

print("\n" + "="*60)
print("VALIDATION")
print("="*60 + "\n")

best_model = YOLO('/content/runs/detect/runs/detect/yolo26n_visdrone/weights/best.pt')

val_metrics = best_model.val()

print(f"mAP50      : {val_metrics.box.map50:.3f}")
print(f"mAP50-95   : {val_metrics.box.map:.3f}")
print(f"Precision  : {val_metrics.box.mp:.3f}")
print(f"Recall     : {val_metrics.box.mr:.3f}")

In [ ]:
from google.colab import files

!zip -r yolo26n_trained.zip /content/runs/detect/runs/detect/yolo26n_visdrone/weights/
!zip -r training_results.zip /content/runs/detect/runs/detect/yolo26n_visdrone/*.png /content/training_curves.png

files.download('yolo26n_trained.zip')
files.download('training_results.zip')